Decoding Dopant-Induced Electronic Modulation in Graphene via Region Resolved Machine Learning of XANES

PCA(Fig.4)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import os

# Load data
file_path = ''
data = pd.read_csv(file_path)

# Extract spectral data and class labels
spectra_data = data.iloc[:, 1:2000].values
data_type = data['type']
b_rate = data['B_rate']
n_rate = data['N_rate']

# Dimensionality reduction via PCA
pca_model = PCA(n_components=2)
pca_result = pca_model.fit_transform(spectra_data)
pca_df = pd.DataFrame(pca_result, columns=['PCA1', 'PCA2'])
pca_df['type'] = data_type
pca_df['B_rate'] = b_rate
pca_df['N_rate'] = n_rate

# Define plot styles
markers = {'type1': 'o', 'type2': '^', 'other': 's'}
colormaps = {'type1': plt.cm.summer, 'type2': plt.cm.winter}
colors = {'other': 'lightgray'}

# Unified colorbar range across both doping types
color_min = min(b_rate.min(), n_rate.min())
color_max = max(b_rate.max(), n_rate.max())

# Plot
plt.figure(figsize=(8, 6))

# Plot type=1 (Boron-doped)
subset_1 = pca_df[pca_df['type'] == 1]
b_scatter = plt.scatter(
    subset_1['PCA1'], subset_1['PCA2'],
    c=subset_1['B_rate'], cmap=colormaps['type1'], marker=markers['type1'],
    edgecolor='#009900', alpha=0.8, label='Boron-doped',
    vmin=color_min, vmax=color_max
)

# Plot type=2 (Nitrogen-doped)
subset_2 = pca_df[pca_df['type'] == 2]
n_scatter = plt.scatter(
    subset_2['PCA1'], subset_2['PCA2'],
    c=subset_2['N_rate'], cmap=colormaps['type2'], marker=markers['type2'],
    edgecolor='#008cff', alpha=0.8, label='Nitrogen-doped',
    vmin=color_min, vmax=color_max
)

# Plot all other types (Pristine)
subset_other = pca_df[(pca_df['type'] != 1) & (pca_df['type'] != 2)]
plt.scatter(
    subset_other['PCA1'], subset_other['PCA2'],
    c=colors['other'], marker=markers['other'], edgecolor='#eb4d00',
    alpha=0.6, label='Pristine'
)

# Colorbar for B_rate
cbar_b = plt.colorbar(b_scatter, fraction=0.03, pad=0.08, aspect=40)
cbar_b.set_label('Boron concentration', fontsize=16, labelpad=10)

# Colorbar for N_rate
cbar_n = plt.colorbar(n_scatter, fraction=0.03, pad=0.08, aspect=40)
cbar_n.set_label('Nitrogen concentration', fontsize=16, labelpad=8)

# Legend and layout
plt.legend(loc='upper right', bbox_to_anchor=(1, 1), fontsize=14)
plt.xlabel('PCA1', fontsize=20)
plt.ylabel('PCA2', fontsize=20)
plt.tick_params(axis='both', labelsize=14)
plt.tight_layout()
plt.show()

Classification(Fig.5)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split

# Load data
file_path = ''
data = pd.read_csv(file_path)

# Define feature ranges
feature_ranges = {
    'PI': data.iloc[:, 499:661].values,
    'SIGMA': data.iloc[:, 661:819].values,
    'Arb': data.iloc[:, 499:819].values,
    'Post': data.iloc[:, 819:1263].values,
    'Global': data.iloc[:, 499:1263].values
}

# Extract PI and SIGMA scalar features
if 'PI' in data.columns and 'SIGMA' in data.columns:
    pi_sigma = data[['PI', 'SIGMA']].values
else:
    print("PI or SIGMA column not found in the data. Please check the CSV file!")
    exit()

# Bin B_rate and N_rate into discrete classes
y_b_rate = data['B_rate'].values
y_n_rate = data['N_rate'].values
bins = [0, 1.39, 2.78, 4.17, 5.56, 6.95, np.inf]
y_b_rate_binned = np.digitize(y_b_rate, bins) - 1
y_n_rate_binned = np.digitize(y_n_rate, bins) - 1

# Store results
results = []

# Cross-validation: compute accuracy and weighted F1
def cross_val_accuracy_weighted_f1(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    accuracies = []
    weighted_f1s = []

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        rf = RandomForestClassifier(n_estimators=100, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        acc = accuracy_score(y_val, y_pred)
        w_f1 = f1_score(y_val, y_pred, average='weighted')

        accuracies.append(acc)
        weighted_f1s.append(w_f1)

    return {
        'accuracy_mean': np.mean(accuracies),
        'accuracy_std': np.std(accuracies),
        'weighted_f1_mean': np.mean(weighted_f1s),
        'weighted_f1_std': np.std(weighted_f1s)
    }

# Iterate over all feature ranges
for name, base_X in feature_ranges.items():
    # Append PI and SIGMA scalar features to current feature set
    X = np.hstack((base_X, pi_sigma))

    # Compute cross-validation results for B_rate and N_rate
    results_b_weighted = cross_val_accuracy_weighted_f1(X, y_b_rate_binned)
    results_n_weighted = cross_val_accuracy_weighted_f1(X, y_n_rate_binned)

    # Record results
    results.append([
        name,
        results_b_weighted['accuracy_mean'], results_b_weighted['accuracy_std'],
        results_b_weighted['weighted_f1_mean'], results_b_weighted['weighted_f1_std'],
        results_n_weighted['accuracy_mean'], results_n_weighted['accuracy_std'],
        results_n_weighted['weighted_f1_mean'], results_n_weighted['weighted_f1_std']
    ])

    # Plot bar chart
def plot_highest_acc_f1_bars(
    title,
    cv_dicts_highest,
    group_labels,
    color_list_boron=None,
    color_list_nitrogen=None,
    alpha=0.8,
    bar_width=0.1,
    spacing=0.2,
    inner_spacing=0.05
):
    """
    Plot grouped bar charts of highest Accuracy and Weighted F1 for multiple groups.
    Each group contains two bars: Accuracy and Weighted F1.
    Values are annotated above each bar (3 decimal places), font size=16.

    Parameters:
    - title: Chart title
    - cv_dicts_highest: List of dicts, each containing accuracy_highest and weighted_f1_highest
    - group_labels: Label for each group
    - color_list_boron: Colors for the Boron group (must contain 2 colors: Accuracy, Weighted F1)
    - color_list_nitrogen: Colors for the Nitrogen group (must contain 2 colors: Accuracy, Weighted F1)
    - alpha: Bar transparency
    - bar_width: Width of each bar
    - spacing: Spacing between groups
    """
    n_groups = len(cv_dicts_highest)
    if color_list_boron is None:
        color_list_boron = ["#009900", "#66c2a5"]
    if color_list_nitrogen is None:
        color_list_nitrogen = ["#008cff", "#667ae9"]
    if len(color_list_boron) < 2 or len(color_list_nitrogen) < 2:
        raise ValueError("Each color list must contain at least 2 colors.")

    all_colors = [color_list_boron, color_list_nitrogen]

    x_positions = []
    all_means = []
    labels = ["Accuracy", "F1 Score"]

    for i, cv_dict in enumerate(cv_dicts_highest):
        accuracy_highest = cv_dict['accuracy_highest']
        weighted_f1_highest = cv_dict['weighted_f1_highest']
        means = [accuracy_highest, weighted_f1_highest]
        group_x_positions = [
            i * (2 * bar_width + spacing) + j * (bar_width + inner_spacing)
            for j in range(2)
        ]
        x_positions.extend(group_x_positions)
        all_means.extend(means)

    fig, ax = plt.subplots(figsize=(6, 6))

    for i in range(len(all_means)):
        group_index = i // 2
        bar_color = all_colors[group_index][i % 2]
        ax.bar(
            x_positions[i],
            all_means[i],
            width=bar_width,
            color=bar_color,
            alpha=alpha,
            capsize=5,
            label=f"{group_labels[group_index]}" if i % 2 == 0 else None
        )
        # Annotate value above each bar
        ax.text(
            x_positions[i],
            all_means[i] + 0.02,
            f"{all_means[i]:.3f}",
            ha='center', va='bottom',
            fontsize=20
        )

    # Set x-axis ticks and labels
    ax.set_xticks(x_positions)
    ax.set_xticklabels(labels * n_groups, fontsize=16, rotation=45)
    ax.set_ylim([0, 1.05])
    ax.set_ylabel("Figure of Merit", fontsize=20)
    ax.legend(
        fontsize=16,
        loc="lower center",
        bbox_to_anchor=(0.52, 0.03),
        ncol=3,
        frameon=True,
        framealpha=0.9,
        edgecolor="black",
        columnspacing=6
    )

    plt.tight_layout()
    plt.show()

# Compile results into DataFrame and save to Excel
results_df = pd.DataFrame(results, columns=[
    'Feature Range',
    'Boron Accuracy Mean', 'Boron Accuracy Std', 'Boron F1 Mean', 'Boron F1 Std',
    'Nitrogen Accuracy Mean', 'Nitrogen Accuracy Std', 'Nitrogen F1 Mean', 'Nitrogen F1 Std'
])
results_file = 'classification_results.xlsx'
results_df.to_excel(results_file, index=False)

print("Analysis complete. Results saved to:", results_file)

Regression1 (mean bond length)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# Load data
file_path = 'data.csv'
data = pd.read_csv(file_path)

#data = data[data['type'].isin([0, 1, 2])]

# Extract features and labels
X = data.iloc[:, 499:661].values  # 499, 661, 819, 1263

# Append PI and SIGMA columns to features
if 'PI' in data.columns and 'SIGMA' in data.columns:
    pi_sigma = data[['PI', 'SIGMA']].values
    X = np.hstack((X, pi_sigma))
else:
    print("PI or SIGMA column not found in the data. Please check the CSV file!")
    exit()

y = data['mbl'].values          # "bader" column as label

# Map type column to colors
color_map = {0: '#eb4d00', 1: '#009900', 2: '#008cff'}
data['color'] = data['type'].map(color_map)

# Split into train and test sets, preserving color assignments
X_train, X_test, y_train, y_test, colors_train, colors_test = train_test_split(
    X, y, data['color'].values, test_size=0.2, random_state=42
)

# Initialize random forest regression model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')

# Print cross-validation results
print("Cross-Validation R² Scores: ", cv_scores)
print("Mean R²: ", np.mean(cv_scores))
print("Standard Deviation of R²: ", np.std(cv_scores))

# Train model on full training set
model.fit(X_train, y_train)

# Generate predictions
y_train_pred = model.predict(X_train)  # Predictions on training set
y_pred = model.predict(X_test)         # Predictions on test set

# Compute R², MSE, and MAE
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# Print evaluation metrics
print("R²: ", r2)
print("MSE: ", mse)
print("MAE: ", mae)

# Custom legend mapping
custom_legend = {
    '#eb4d00': 'Pristine (test)',
    '#009900': 'BG (test)',
    '#008cff': 'NG (test)',
    '#b5a26d': 'Training Set'
}

# Parity plot (full range)
plt.figure(figsize=(6, 6))
# Plot training set (golden, alpha=0.6, square markers)
plt.scatter(y_train, y_train_pred, color='#b5a26d', alpha=0.6, label='Training Set', marker='s')

# Plot test set colored by type
for i, color in enumerate(colors_test):
    plt.scatter(y_test[i], y_pred[i], color=color, alpha=0.6)
plt.plot([1.400, 1.510], [1.400, 1.510], color='red', linestyle='--', linewidth=1)
plt.xlim(1.400, 1.510)
plt.ylim(1.400, 1.510)
plt.tick_params(axis='both', labelsize=14)
plt.xlabel('DFT (Å)', fontsize=20)
plt.ylabel('Predicted (Å)', fontsize=20)
plt.title("Mean Nearest-Neighbor Distance \n Regression Performance", fontsize=20)

# Build legend entries
legend_added = set()  # Track which colors have been added
# Add test set entries by type
for color in custom_legend:
    if color != '#ffd700' and color in colors_test and color not in legend_added:
        plt.scatter([], [], color=color, label=custom_legend[color])  # Dummy point for legend
        legend_added.add(color)
        
plt.legend(fontsize=16)
plt.show()

Regression (bader charge)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# Load data
file_path = 'data.csv'
data = pd.read_csv(file_path)

#data = data[data['type'].isin([0, 1, 2])]

# Extract features and labels
X = data.iloc[:, 499:661].values  # 499, 661, 819, 1263

# Append PI and SIGMA columns to features
if 'PI' in data.columns and 'SIGMA' in data.columns:
    pi_sigma = data[['PI', 'SIGMA']].values
    X = np.hstack((X, pi_sigma))
else:
    print("PI or SIGMA column not found in the data. Please check the CSV file!")
    exit()

y = data['bader'].values          # "bader" column as label

# Map type column to colors
color_map = {0: '#eb4d00', 1: '#009900', 2: '#008cff'}
data['color'] = data['type'].map(color_map)

# Split into train and test sets, preserving color assignments
X_train, X_test, y_train, y_test, colors_train, colors_test = train_test_split(
    X, y, data['color'].values, test_size=0.2, random_state=42
)

# Initialize random forest regression model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')

# Print cross-validation results
print("Cross-Validation R² Scores: ", cv_scores)
print("Mean R²: ", np.mean(cv_scores))
print("Standard Deviation of R²: ", np.std(cv_scores))

# Train model on full training set
model.fit(X_train, y_train)

# Generate predictions
y_train_pred = model.predict(X_train)  # Predictions on training set
y_pred = model.predict(X_test)         # Predictions on test set

# Compute R², MSE, and MAE
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# Print evaluation metrics
print("R²: ", r2)
print("MSE: ", mse)
print("MAE: ", mae)

# Custom legend mapping
custom_legend = {
    '#eb4d00': 'Pristine (test)',
    '#009900': 'BG (test)',
    '#008cff': 'NG (test)',
    '#b5a26d': 'Training Set'
}

# Parity plot (full range)
plt.figure(figsize=(6, 6))
# Plot training set (golden, alpha=0.6, square markers)
plt.scatter(y_train, y_train_pred, color='#b5a26d', alpha=0.6, label='Training Set', marker='s')

# Plot test set colored by type
for i, color in enumerate(colors_test):
    plt.scatter(y_test[i], y_pred[i], color=color, alpha=0.6)

plt.plot([-1.5, 2], [-1.5, 2], color='red', linestyle='--', linewidth=1)  # Diagonal reference line
plt.xlim(-1.5, 2)
plt.ylim(-1.5, 2)
plt.tick_params(axis='both', labelsize=14)
plt.xlabel('DFT', fontsize=20)
plt.ylabel('Predicted', fontsize=20)
plt.title("Mean Bader Charge \n Regression Performance", fontsize=20)

# Build legend entries
legend_added = set()  # Track which colors have been added
# Add test set entries by type
for color in custom_legend:
    if color != '#ffd700' and color in colors_test and color not in legend_added:
        plt.scatter([], [], color=color, label=custom_legend[color])  # Dummy point for legend
        legend_added.add(color)

plt.legend(fontsize=16)
plt.show()